# Tutorial 1 - First steps with ParlaMint dataset

Welcome to this **hands-on tutorial exploring the ParlaMint dataset**, a rich multilingual corpus of European parliamentary debates. This tutorial is designed for researchers, data scientists and political science enthusiasts who want to analyze parliamentary proceedings across different European countries. 

The **ParlaMint dataset** is a comprehensive and multilingual corpus of transcribed speeches from different European parliaments. It serves as a valuable resource for computational social science and digital humanities research. 

At its core, ParlaMint contains the full text of parliamentary speeches annotated with rich metadata, including the speaker's role (e.g. *Member of Parliament*), political party affiliation, date of the session and the length of each speech. This foundational data was significantly expanded with **CAP categories** and **sentiment scores**.

First, the ParlaCAP component categorizes all available speech segments into a specific policy domain from the **Comparative Agendas Project (CAP)**, e.g. healthcare, education or foreign affairs. 

Second, the **ParlaSent** extension provides detailed **sentiment scores** for each segment, allowing researchers to analyze the emotional tone of the debates.

The dataset's power lies in its comparative design. It covers **multiple countries** and spans different time periods, enabling cross-parliament analysis. This makes ParlaMint very useful for **comparative political research**, allowing users to systematically study the differences in e.g. political discourse or policy priorities, across nations. Besides that, its structure makes it ideal for technical applications like **topic modeling** and **sentiment analysis** on a large scale.

*Source: Erjavec, Tomaž; et al. (2025). Multilingual comparable corpora of parliamentary debates ParlaMint 5.0. Slovenian language resource repository CLARIN.SI. ISSN 2820-4042. http://hdl.handle.net/11356/2004*


**Importing Libraries**

We only need a few tools here.


In [1]:
import pandas as pd
import requests
import io
import time

# Dataset schema and extent

Let's call the API's `variables` endpoint that will tell us more about the dataset:

In [2]:
url = "https://parlacap.ipipan.waw.pl/api/v1/"
headers = {"X-API-Key": open("parlacapi_key").read()}
response = requests.get(url + "variables", headers=headers)
if response.status_code != 200:
    raise Exception(f"Got weird response code: {response.content}")
variables = pd.DataFrame(response.json())
variables


,name,type,description,stats
0,cap_category,TEXT,Predicted CAP category.,"{'count': 7982341, 'unique_count': 23, 'null_c..."
1,cap_prob,REAL,CAP category prediction probability.,"{'count': 7982341, 'unique_count': 880, 'null_..."
2,date,TEXT,Date of the session.,"{'count': 7982752, 'unique_count': 5723, 'null..."
3,id,INTEGER,Primary key,None
4,lang,TEXT,Language of the transcript,"{'count': 7982752, 'unique_count': 30, 'null_c..."
5,parlacap_id,TEXT,Original document ID from ParlaCap,None
6,parlamint_id,TEXT,Original speech ID from ParlaMint.,None
7,parlamint_text_id,TEXT,Original document ID from ParlaMint.,None
8,parliament,TEXT,"Parliament code (e.g. BA, ES-GA, ...)","{'count': 7982752, 'unique_count': 28, 'null_c..."
9,party_orientation,TEXT,"Political orientation (e.g., left, right).","{'count': 6582010, 'unique_count': 44, 'null_c..."


This tells us the available columns (or attributes) of our instances, what they are called in the database, what type the values in the columns are (e.g. text, integer...), and where applicable, also some statistics. 

Let's investigate the `lang` column some more:

In [3]:
variables.loc[variables.name == "lang", "stats"].values[0]

{'count': 7982752,
 'unique_count': 30,
 'null_count': 0,
 'top': 'French',
 'freq': 795637,
 'top10': [{'value': 'French', 'freq': 795637},
  {'value': 'Turkish', 'freq': 739744},
  {'value': 'Dutch', 'freq': 699925},
  {'value': 'English', 'freq': 671038},
  {'value': 'Croatian', 'freq': 504334},
  {'value': 'Ukrainian', 'freq': 402731},
  {'value': 'Danish', 'freq': 398610},
  {'value': 'Greek', 'freq': 342274},
  {'value': 'Norwegian bokmål', 'freq': 334636},
  {'value': 'Serbian', 'freq': 316069}]}

This tells us that the whole database has 7982752 non-null values in this column, there are 30 unique values in it, and since `null_value` is 0, we know that all the rows in the database have language assigned to them.

In addition, we can also see that the most frequent language is French, and we can see the breakdown of top 10 most frequent values in this column.

# Download the data

Let us download the data  we will use in subsequent tutorials. In the cell below, we ask the API to return only the columns we need, and to also do some filtering for us before sending us the data. The API uses an **asynchronous export system** for Parquet, XLSX, and RDS formats: we submit a request for the data, and if the dataset is large, the API returns a job ID. We then **periodically poll the API** to check if the export is ready, and download it once it's complete. Depending on the server load, this can take a few minutes.

In [4]:
url = "https://parlacap.ipipan.waw.pl/api/v1/"
headers = {"X-API-Key": open("parlacapi_key").read()}

# Define the filter for our sample
filter_data = {
    "columns": [
        "cap_category",
        "date",
        "parliament",
        "party_status",
        "sent_logit",
        "sent3_category",
        "speaker_mp",
        "speaker_party_name",
        "speaker_role",
        "word_count",
    ],
}

print("Submitting data request to API...")
response = requests.post(
    url + "download",
    json=filter_data,
    headers=headers,
    params={"format": "parquet"},
    stream=True,
)

if response.status_code == 202:
    # Job is being processed asynchronously
    job = response.json()
    print(f"Export job {job['job_id']} started, polling for status...")

    while True:
        status_response = requests.get(
            url + job["status_url"].lstrip("/"), headers=headers
        ).json()

        if status_response["status"] == "ready":
            print("Data is ready for download!")
            break
        if status_response["status"] == "error":
            raise Exception(f"Export failed: {status_response.get('detail')}")

        time.sleep(10)

    print(f"Export job {job['job_id']} succeeded, download starting...")
    data_response = requests.get(url + job["download_url"].lstrip("/"), headers=headers)
    data_response.raise_for_status()

    # Load into DataFrame
    filtered_all = pd.read_parquet(io.BytesIO(data_response.content))

    print(f"Download complete! Retrieved {len(filtered_all)} rows.")

elif response.status_code == 200:
    filtered_all = pd.read_parquet(io.BytesIO(response.content))
    print(f"Direct download complete! Retrieved {len(filtered_all)} rows.")

else:
    raise Exception(
        f"Got unexpected response code: {response.status_code}, {response.content}"
    )

# Rename columns for consistency
filtered_all = filtered_all.rename(
    columns={"cap_category": "CAP_category", "parliament": "country"}
)
filtered_all = filtered_all[
    (~filtered_all["CAP_category"].isin(["Mix", "Other", None, "none", "None"]))
    & (filtered_all["party_status"].notna())
    & (filtered_all["sent_logit"].notna())
    & (filtered_all["date"].notna())
    & (filtered_all["speaker_role"] == "Regular")
    & (filtered_all["speaker_mp"] == "MP")
]

if filtered_all.empty:
    raise ValueError("No data came back. Check the API key.")

# Clean the table a little bit.
filtered_all["date"] = pd.to_datetime(filtered_all["date"], errors="coerce")
filtered_all = filtered_all.dropna(subset=["CAP_category", "sent_logit"])
filtered_all.to_parquet("data.parquet")

print(f"Final dataset shape: {filtered_all.shape}")

Submitting data request to API...
Export job 0cf1efb3-40a6-4b57-bdd3-53f9d1ded7c2 started, polling for status...
Data is ready for download!
Export job 0cf1efb3-40a6-4b57-bdd3-53f9d1ded7c2 succeeded, download starting...
Download complete! Retrieved 7982752 rows.
Final dataset shape: (1859625, 10)


# Other ways to access the data

For downloading smaller subsets of the data, for instance only one parliament at a time, we can use the `/download` endpoint and request either CSV, TSV, or JSONL. In this case, no waiting is necessary, but if the returned data is too big, we will wait longer than we would for the asynchronously returned data. Note that the data is gzipped, so before we read it, we will need to decompress it.

In the next cell we will request data from the Bosnian parliament, columns `"cap_category", "sent_logit", "parliament", "date"`  . 

In [5]:
import requests, io, gzip
import pandas as pd

url = "https://parlacap.ipipan.waw.pl/api/v1/"
headers = {"X-API-Key": open("parlacapi_key").read()}

# Define the filter for our sample
filter_data = {
    "columns": ["cap_category", "sent_logit", "parliament", "date"],
    "filter": {"column": "parliament", "value": "BA", "operator": "="},
}

response = requests.post(
    url + "download",
    json=filter_data,
    headers=headers,
    params={"format": "jsonl"},
    # stream=True,
)

df = pd.read_json(io.StringIO(gzip.decompress(response.content).decode()), lines=True)
df = df.rename(columns={"cap_category": "CAP_category", "parliament": "country"})
df = df[~df["CAP_category"].isin(["Mix", "Other"])]

**3.2. shape()**

Our next question is: **How much data are we working with?**

The '.shape()' method answers this by outputting a pair of numbers in this format: '(number_of_rows, number_of_columns)'.


The output, e.g. (2000, 4), would tell us that we are analyzing **2000 rows (individual speeches)**, each described by **4 different variables**. 


In [6]:
df.shape

(44534, 4)

### 3.3 df.describe()

We can  use `df.describe()` to ask pandas to describe the dataframe by columns with some aggregating functions like minimum, average, most common value... 


In [7]:
df[["country", "CAP_category", "sent_logit"]].describe(include="all")


,country,CAP_category,sent_logit
count,44534,44534,44534.000000
unique,1,21,NaN
top,BA,Government Operations,NaN
freq,44534,13775,NaN
mean,NaN,NaN,2.259465
std,NaN,NaN,1.048436
min,NaN,NaN,-0.151000
25%,NaN,NaN,1.431000
50%,NaN,NaN,2.253000
75%,NaN,NaN,3.106000


**3.4. unique()**

The 'unique()' method extracts a clean list of all possible values in a column (e.g. 'CAP_category'). It extracts every distinct category without any duplicates or counts.

When we wrap that method in 'pd.Series()', it outputs a list in an easy-to-read format.


In [8]:
pd.Series(df["CAP_category"].unique())

0     Government Operations
1             Law and Crime
2     International Affairs
3              Civil Rights
4            Macroeconomics
5               Immigration
6            Transportation
7                   Culture
8         Domestic Commerce
9                Technology
10                  Housing
11                  Defense
12              Agriculture
13            Foreign Trade
14           Social Welfare
15                   Health
16                   Energy
17             Public Lands
18                Education
19                    Labor
20              Environment
dtype: object

Here, we do the same but instead of 'CAP_category', we look at all unique values in the column 'country'.


In [9]:
pd.Series(df["country"].unique())

0    BA
dtype: object

**3.5. value_counts()**

To understand the data on a deeper level, we use '.value_counts()'. This method **counts how many times each unique value appears** in a column. Also, it shows us what values are the most common.


In [10]:
df["CAP_category"].value_counts()

CAP_category
Government Operations    13775
Macroeconomics            5537
International Affairs     4639
Civil Rights              4499
Law and Crime             4420
Immigration               1805
Defense                   1676
Transportation            1260
Foreign Trade             1037
Domestic Commerce          988
Agriculture                957
Technology                 906
Education                  820
Health                     615
Energy                     460
Labor                      356
Environment                194
Social Welfare             188
Public Lands               178
Housing                    147
Culture                     77
Name: count, dtype: int64

In the case of 'CAP_category', the command above answers the question: **"Which policy topics dominate the parliamentary agenda?"**. The output is a ranked list, showing the most frequently debated topics at the top. 

*The same method can be applied to other columns, such as 'party_orientation' to see the left-right balance of speeches.*


**3.6. Checking for missing values**

Before any analysis, we must veryify that our key variables are complete. The '.isnull()' method checks each value in a column to see if it's *empty*. By combining it with '.values.any()', it outputs a single, clear answer.

In [11]:
(
    df["CAP_category"].isnull().any(),
    df["date"].isnull().any(),
    df["sent_logit"].isnull().any(),
)


(False, False, False)

**Conclusion**

In this first tutorial, we took the essential steps toward working with the ParlaMint dataset. We learned how to:
- **Set up the environment** by installing and importing the necessary Python libraries
- **Load and filter the data efficiently**
- **Explore the data** with foundational 'pandas'-methods such as '.head()', '.shape()', '.describe()', 'unique()' and '.value_counts()' to get a first overview of the structure and content

By the end of this tutorial, you now have a **clean, filtered dataset sample** of parliamentary debates that is ready for deeper analysis. 

In the next tutorial (**Tutorial 2**), we will take this filtered dataset and explore **topic and sentiment distributions** in depth - visualizing which CAP categories dominate parliamentary debates or how sentiment varies by topic and country.
